## Deep Learning Cohort Fertility Prediction

Trains the non-log DL model with rotating jump-off years (1985-2010) and converts predictions to cohort rates.

In [22]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import random

In [23]:
import training_functions
import importlib

importlib.reload(training_functions)

<module 'training_functions' from '/Users/paigepark/Desktop/repos/deep-fert/code/training_functions.py'>

In [ ]:
asfr_training = np.loadtxt('../data/asfr_training.txt')
asfr_test = np.loadtxt('../data/asfr_test.txt')
asfr_all = np.vstack([asfr_training, asfr_test])

# Save data for R scripts
np.savetxt('../data/asfr_1950_to_2015.txt', asfr_all)

print(f"Data: {asfr_all.shape}")
print(f"Years: {int(asfr_all[:,1].min())}-{int(asfr_all[:,1].max())}")
print(f"Ages: {int(asfr_all[:,2].min())}-{int(asfr_all[:,2].max())}")
print(f"Countries: {int(asfr_all[:,0].max()) + 1}")

In [ ]:
JUMP_OFF_YEARS = [1985, 1990, 1995, 2000, 2005, 2010]
YEAR_MIN = 1950
YEAR_MAX = 2015
FORECAST_LEN = 30
STEPS_RATIO = 4.74
BATCH_SIZE = 256
METHOD_NAME = "DL_NonLog"

geo_dim = int(asfr_all[:, 0].max()) + 1
ages = np.arange(int(asfr_all[:, 2].min()), int(asfr_all[:, 2].max()) + 1)
countries = np.arange(geo_dim)

print(f"geo_dim: {geo_dim}")
print(f"Ages: {ages.min()}-{ages.max()}")
print(f"Jump-off years: {JUMP_OFF_YEARS}")

In [ ]:
predicted_rates = {}
observed_rates = {}

for joy in JUMP_OFF_YEARS:
    print(f"\n{'='*50}")
    print(f"Jump-off year: {joy}")
    print(f"{'='*50}")

    # 1. Training data: all countries, all ages, year <= joy
    train_mask = asfr_all[:, 1] <= joy
    train_data = asfr_all[train_mask]
    print(f"Training rows: {train_data.shape[0]}")

    # 2. Validation data: year > joy (up to YEAR_MAX)
    val_mask = asfr_all[:, 1] > joy
    val_data = asfr_all[val_mask]
    print(f"Validation rows: {val_data.shape[0]}")

    if train_data.shape[0] == 0:
        print("No training data, skipping")
        continue

    # 3. Scale steps_per_epoch
    steps_per_epoch = int(train_data.shape[0] * STEPS_RATIO / BATCH_SIZE)
    print(f"Steps per epoch: {steps_per_epoch}")

    # 4. Prep datasets
    train_prepped = training_functions.prep_data(train_data, mode="train", changeratetolog=False)
    val_prepped = training_functions.prep_data(val_data, mode="test", changeratetolog=False)

    # 5. Set seeds and train non-log model
    np.random.seed(42)
    tf.random.set_seed(42)
    random.seed(42)
    os.environ['PYTHONHASHSEED'] = str(42)

    model, val_loss = training_functions.run_deep_model(
        train_prepped, val_prepped, geo_dim,
        epochs=50,
        steps_per_epoch=steps_per_epoch,
        lograte=False
    )
    print(f"Best val loss: {val_loss:.6f}")

    # 6. Forecast grid: all countries, all ages, years JOY+1 to JOY+FORECAST_LEN
    forecast_year_max = joy + FORECAST_LEN
    forecast_years = np.arange(joy + 1, forecast_year_max + 1)
    grid = np.array([(c, y, a) for c in countries for y in forecast_years for a in ages])
    print(f"Forecast grid: {grid.shape[0]} ({len(countries)} countries, {len(forecast_years)} years, {len(ages)} ages)")

    # 7. Predict using same normalization as training_functions.py
    forecast_features = (
        tf.convert_to_tensor((grid[:, 1] - YEAR_MIN) / (YEAR_MAX - YEAR_MIN), dtype=tf.float32),
        tf.convert_to_tensor(grid[:, 2], dtype=tf.float32),
        tf.convert_to_tensor(grid[:, 0], dtype=tf.float32),
    )
    preds = model.predict(forecast_features).flatten()

    # 8. Store predicted period rates
    pred_df = pd.DataFrame({
        'Country': grid[:, 0].astype(int),
        'Year': grid[:, 1].astype(int),
        'Age': grid[:, 2].astype(int),
        'Rate': preds,
    })
    predicted_rates[joy] = pred_df

    # 9. Store observed period rates (year <= joy)
    obs_df = pd.DataFrame({
        'Country': train_data[:, 0].astype(int),
        'Year': train_data[:, 1].astype(int),
        'Age': train_data[:, 2].astype(int),
        'Rate': train_data[:, 3],
    })
    observed_rates[joy] = obs_df

    # 10. Save model
    model.save(f"../models/dl_cohort_nonlog_joy{joy}.keras")
    print(f"Model saved: ../models/dl_cohort_nonlog_joy{joy}.keras")

print("\nAll jump-off years complete!")

In [27]:
def period_to_cohort(period_df):
    df = period_df.copy()
    df['Year'] = df['Year'] - df['Age']  # cohort_birth_year = period_year - age
    return df

In [ ]:
pred_cohort_dfs = []

for joy in JUMP_OFF_YEARS:
    # Observed period rates (year <= JOY)
    obs_period = observed_rates[joy]

    # DL-predicted period rates (JOY+1 to JOY+FORECAST_LEN)
    pred_period = predicted_rates[joy]

    # Combine observed + predicted period rates
    combined_period = pd.concat([obs_period, pred_period], ignore_index=True)

    # Convert to cohort
    cohort_df = period_to_cohort(combined_period)

    # Add metadata columns
    cohort_df['JumpOffYear'] = joy
    cohort_df['Method'] = METHOD_NAME
    cohort_df['Key'] = cohort_df['Method'] + '_' + cohort_df['Country'].astype(str)

    pred_cohort_dfs.append(cohort_df)

predCASFR = pd.concat(pred_cohort_dfs, ignore_index=True)
print(f"predCASFR shape: {predCASFR.shape}")
print(f"Columns: {list(predCASFR.columns)}")
predCASFR.head()

In [ ]:
obs_cohort_dfs = []

# Full observed period data
all_obs_df = pd.DataFrame({
    'Country': asfr_all[:, 0].astype(int),
    'Year': asfr_all[:, 1].astype(int),
    'Age': asfr_all[:, 2].astype(int),
    'Rate': asfr_all[:, 3],
})

for joy in JUMP_OFF_YEARS:
    # Convert all observed period data to cohort
    cohort_df = period_to_cohort(all_obs_df)

    # Add metadata columns
    cohort_df['JumpOffYear'] = joy
    cohort_df['Method'] = METHOD_NAME
    cohort_df['Key'] = cohort_df['Method'] + '_' + cohort_df['Country'].astype(str)

    obs_cohort_dfs.append(cohort_df)

obsCASFR = pd.concat(obs_cohort_dfs, ignore_index=True)
print(f"obsCASFR shape: {obsCASFR.shape}")
print(f"Columns: {list(obsCASFR.columns)}")
obsCASFR.head()

In [ ]:
predCASFR.to_csv('../data/dl_forecasts_cohort_v1.csv', index=False)
obsCASFR.to_csv('../data/dl_obs_cohort_v1.csv', index=False)

print(f"Saved: ../data/dl_forecasts_cohort_v1.csv ({predCASFR.shape[0]} rows)")
print(f"Saved: ../data/dl_obs_cohort_v1.csv ({obsCASFR.shape[0]} rows)")